<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-12d-multi-gpu-serving-parallelism-and-cost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Discussion 12 (continued) - Serving a model that does not fit on one GPU: tensor, pipeline and expert parallelism, and GPU cost

How many GPUs a model needs, what tensor parallelism buys and costs on NVLink versus PCIe, why pipeline parallelism has a bubble, why expert-parallel (mixture-of-experts) models behave differently at batch 1 and batch 32, and how to pick the cheapest configuration that meets a latency target.

> ## ⚠️ Multi-GPU needs paid hardware
> Everything here is **arithmetic and simulation that runs free on the Colab CPU**. Real multi-GPU serving needs an instance with **2-8 GPUs** - Colab only ever provides one - for example rented with the free-credit route from Chapter 15. The final part only prints commands and a topology check; nothing graded depends on it. Prices and availability change.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate.

*Original code written for this course. Techniques follow the Megatron-LM paper (Shoeybi et al., 2019, arXiv:1909.08053), GPipe, and the documentation of vLLM and TensorRT-LLM; interconnect and memory figures are public datasheet values, quoted as approximations.*

In [ ]:
%pip install -q numpy

In [ ]:
import math, numpy as np
np.random.seed(0)

## Part 1 - How many GPUs does the model need?

The first-order answer is memory: weights plus the KV cache for all concurrent requests, divided by usable GPU memory. We keep 10% headroom for activations and buffers. KV bytes per token come from the earlier discussion: `2 x layers x kv_heads x head_dim x bytes`.

In [ ]:
def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_=2):
    return 2 * layers * kv_heads * head_dim * bytes_

def min_gpus(weights_gb, kv_gb, gpu_gb, headroom=0.10):
    need = weights_gb + kv_gb
    return math.ceil(need / (gpu_gb * (1 - headroom)))

# a 70B model with grouped-query attention: 80 layers, 8 KV heads, head_dim 128
kv_tok = kv_bytes_per_token(80, 8, 128)                       # 327,680 bytes (fp16 cache)
users, ctx = 32, 4096
kv_gb = kv_tok * users * ctx / 1e9
print(f"KV cache for {users} users x {ctx} tokens: {kv_gb:.1f} GB (fp16), {kv_gb/2:.1f} GB (fp8)")
print(f"{'weights':>10} {'KV':>6} {'24GB (L4)':>10} {'40GB':>6} {'80GB (A100/H100)':>17}")
for label, w, kv in (("fp16 140GB", 140, kv_gb), ("fp8 70GB", 70, kv_gb/2), ("int4 ~36GB", 36, kv_gb/2)):
    print(f"{label:>10} {kv:>5.0f}G {min_gpus(w, kv, 24):>10} {min_gpus(w, kv, 40):>6} {min_gpus(w, kv, 80):>17}")
print("\n(minimum GPU count by memory alone; in practice use a power of two that divides the attention heads - 2, 4 or 8)")

**Read it.** Lower precision does double duty: it can cut the GPU count (fewer GPUs to rent and fewer GPUs' worth of synchronisation) *and* it raises the decode ceiling. Memory only gives the **minimum**; whether to use more GPUs than that is a latency and cost question, answered below.

## Part 2 - Tensor parallelism: faster per token, but you pay a toll twice per layer

Tensor parallelism (the Megatron-LM idea) splits every weight matrix across `p` GPUs. Each GPU streams only `1/p` of the weights per token, so the memory-bound decode step gets close to `p` times faster **in the weight-streaming term**, but each transformer layer needs two all-reduces of the activations (after attention and after the MLP) to combine partial results. For small decode batches those messages are tiny, so the cost is dominated by **latency**, not bandwidth - and a slow interconnect (PCIe) hurts far more than NVLink.

Model below: `t_step = weights_bytes / (p x memory_bandwidth) + 2 x layers x t_allreduce`, with a ring all-reduce time of `latency + 2(p-1)/p x message_bytes / link_bandwidth`. The latency and bandwidth values are **illustrative round numbers of the right order of magnitude** (NVLink: a few microseconds, hundreds of GB/s; PCIe: tens of microseconds, ~30 GB/s per direction), not measurements.

In [ ]:
LINKS = {"NVLink": {"lat_us": 8,  "bw_gbs": 400},       # illustrative
         "PCIe":   {"lat_us": 25, "bw_gbs": 28}}        # illustrative

def allreduce_s(msg_bytes, p, link):
    if p == 1: return 0.0
    L = LINKS[link]
    return L["lat_us"] * 1e-6 + 2 * (p - 1) / p * msg_bytes / (L["bw_gbs"] * 1e9)

def decode_step_s(weights_gb, mem_bw_gbs, layers, hidden, batch, p, link):
    stream = weights_gb * 1e9 / (p * mem_bw_gbs * 1e9)
    comm = 2 * layers * allreduce_s(batch * hidden * 2, p, link)
    return stream + comm, stream, comm

W, BW, LAYERS, HIDDEN, BATCH = 70, 3350, 80, 8192, 16       # 70B fp8 weights, H100-class memory bandwidth
print(f"70B fp8 on H100-class GPUs, batch {BATCH}.  (per-user tokens/s = 1/step; aggregate = batch/step)")
print(f"{'link':>7} {'GPUs':>5} {'step ms':>8} {'stream':>8} {'comm':>7} {'per-user':>9} {'aggregate':>10} {'agg/GPU':>8}")
for link in LINKS:
    for p in (1, 2, 4, 8):
        t, s, c = decode_step_s(W, BW, LAYERS, HIDDEN, BATCH, p, link)
        print(f"{link:>7} {p:>5} {t*1e3:>8.2f} {s*1e3:>8.2f} {c*1e3:>7.2f} {1/t:>9.0f} {BATCH/t:>10.0f} {BATCH/t/p:>8.0f}")

**Read it.** On NVLink, going from 1 to 8 GPUs keeps cutting per-user latency (20.9 ms to 4.1 ms per step), but the aggregate throughput *per GPU* falls from 766 to 491 tokens/s as the toll grows. On PCIe the toll is several times larger: at 8 GPUs communication takes about 6.6 ms of a 9.2 ms step, and each GPU delivers only about 217 tokens/s - less than half of NVLink's. That is why multi-GPU nodes sold for serving have NVLink, and why "just add GPUs" is not a strategy. (The model ignores overlap of communication with compute, which real engines use to hide part of the cost - treat it as the shape, not the number. Measure on your hardware.)

## Part 3 - Pipeline parallelism and the bubble

Pipeline parallelism cuts the model by *layers*: GPU 1 holds the first block of layers, GPU 2 the next, and so on, with only one activation tensor passed between neighbours. Traffic is small, so it suits slow links and multi-node setups. The cost is the **pipeline bubble**: when `p` stages process `m` micro-batches, the fraction of time stages sit idle is `(p - 1) / (m + p - 1)` (the GPipe analysis). More micro-batches shrink it.

In [ ]:
def bubble(p, m): return (p - 1) / (m + p - 1)
print(f"{'stages':>7} " + "".join(f"{'m='+str(m):>8}" for m in (1, 4, 8, 16, 64)))
for p in (2, 4, 8):
    print(f"{p:>7} " + "".join(f"{bubble(p, m):>8.0%}" for m in (1, 4, 8, 16, 64)))

In serving, a request's tokens must still pass through every stage in turn, so pipeline parallelism **adds latency per token** and pays off only through throughput (many requests in flight fill the stages). Rule of thumb: tensor-parallel inside a node (fast link), pipeline-parallel across nodes (slow link), and the two combined when a model spans several nodes.

## Part 4 - Expert parallelism: why mixture-of-experts behaves differently at batch 1 and batch 32

A mixture-of-experts (MoE) layer has `E` expert networks and routes each token to its top `k`. At batch 1, decode reads only `k` experts per layer - far fewer weights than a dense model of the same total size, so it is fast. But with a batch of `B` tokens, each choosing experts, the expected number of **distinct experts touched** is `E x (1 - (1 - k/E)^B)` (assuming uniform routing) - which approaches `E` quickly. The memory-bandwidth advantage evaporates at realistic batch sizes, while all `E` experts still have to live in memory.

In [ ]:
def experts_touched(E, k, B): return E * (1 - (1 - k / E) ** B)
E, k = 64, 2
print(f"MoE layer with {E} experts, top-{k} routing (uniform routing assumed)")
print(f"{'batch':>6} {'experts read':>13} {'fraction of all expert weights read':>38}")
for B in (1, 4, 8, 16, 32, 64):
    n = experts_touched(E, k, B)
    print(f"{B:>6} {n:>13.1f} {n/E:>38.0%}")

**Read it.** Expert parallelism places different experts on different GPUs and routes tokens between them (an all-to-all exchange). It exists because the experts must be stored *somewhere*; it does not make batch-32 decode read less memory than a dense model of the same size. Real routing is not uniform, so real numbers sit somewhat below this idealisation, but the direction of the effect is why MoE serving is usually a throughput game at large batches, and a poor fit for tiny deployments.

## Part 5 - The cheapest configuration that meets the latency target

Throughput per dollar is not maximised by the most GPUs. Pick a per-user latency target, find the smallest configurations that meet it, then compare cost per million tokens. GPU prices below are **placeholders** - replace them with your cloud's current price.

In [ ]:
PRICE_PER_GPU_HR = 4.0                    # placeholder USD per GPU-hour
UTIL = 0.6
TARGET_ITL_MS = 12                        # per-user inter-token latency target

def usd_per_mtok(p, link, batch):
    t, _, _ = decode_step_s(W, BW, LAYERS, HIDDEN, batch, p, link)
    agg = batch / t
    return p * PRICE_PER_GPU_HR / (agg * 3600 * UTIL) * 1e6, t * 1e3, agg

print(f"70B fp8, target ITL <= {TARGET_ITL_MS} ms, GPU placeholder ${PRICE_PER_GPU_HR}/h, utilisation {UTIL:.0%}")
print(f"{'link':>7} {'GPUs':>5} {'batch':>6} {'ITL ms':>7} {'agg tok/s':>10} {'$/1M tok':>9}  meets target?")
best = None
for link in LINKS:
    for p in (2, 4, 8):
        for batch in (8, 32, 64):
            c, itl, agg = usd_per_mtok(p, link, batch)
            ok = itl <= TARGET_ITL_MS
            if ok and (best is None or c < best[0]): best = (c, link, p, batch)
            if batch == 32: print(f"{link:>7} {p:>5} {batch:>6} {itl:>7.1f} {agg:>10.0f} {c:>9.2f}  {'yes' if ok else 'no'}")
print("\ncheapest option meeting the target across the grid (batch 8/32/64):", best)

**The method, not the numbers:** (1) compute the memory minimum; (2) list the configurations that meet the latency target on *your* interconnect; (3) among those, choose the cheapest per million tokens at a *realistic* utilisation; (4) re-measure with the benchmark harness from the TensorRT-LLM discussion. Quantisation, a smaller model, speculative decoding and better batching often beat adding GPUs.

## Part 6 - OPTIONAL, PAID MULTI-GPU: topology check and the commands

**Requires a machine with 2 or more NVIDIA GPUs.** The cell prints your GPU topology (how the GPUs are linked: `NV#` means NVLink, `PIX`/`PHB`/`SYS` mean PCIe paths) and the commands that start a tensor-parallel server with each engine. It never starts a server by itself.

In [ ]:
import shutil, subprocess
n_gpu = 0
if shutil.which("nvidia-smi"):
    try: n_gpu = len(subprocess.check_output(["nvidia-smi", "-L"], text=True).strip().splitlines())
    except Exception: pass
print("GPUs visible:", n_gpu)
if n_gpu >= 2:
    topo = subprocess.run(["nvidia-smi", "topo", "-m"], capture_output=True, text=True)
    print(topo.stdout if topo.returncode == 0 else "topology matrix not available on this platform (Linux: nvidia-smi topo -m)")
    print(f"vllm serve <model> --tensor-parallel-size {n_gpu}")
    print(f"trtllm-serve <model> --backend pytorch --tp_size {n_gpu}")
    print("then benchmark each with the harness from the TensorRT-LLM discussion at several concurrency levels and compare with Part 2's predictions.")
else:
    print("Fewer than 2 GPUs - nothing to check. Parts 1-5 are the free, offline path.")

## What to hand in if you ran Part 6

GPU model and count, the topology matrix, the engine and flag used, the measured per-user inter-token latency and aggregate throughput at two or three concurrency levels, and one paragraph comparing them with Part 2's prediction (where was the model wrong, and why?).